# Basic RAG Pipeline: Retrieval-Augmented Generation from Scratch

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline from scratch
using only fundamental libraries — no frameworks. You'll see exactly what happens
at each step: embedding, retrieval, and generation.

**What you'll learn:**
- How to chunk documents and generate OpenAI embeddings
- How cosine similarity finds relevant text chunks
- How to augment a prompt with retrieved context
- How RAG reduces hallucinations vs. answering from memory alone
- (Optional) How to visualize similarity scores across all chunks

**Prerequisites:** Basic Python, familiarity with APIs.

## Install Packages

In [ ]:
!pip install -q openai==2.8.1 google-genai==1.69.0 scikit-learn pandas numpy matplotlib tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


In [ ]:
import os

try:
    # In Google Colab: add your keys via the Secrets panel (key icon in sidebar)
    # and name them 'OPENAI_API_KEY' and 'GOOGLE_API_KEY'.
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("API keys loaded from Colab Secrets.")
except ImportError:
    # Running locally — keys are read from environment variables.
    # Set them in your shell or a .env file before running.
    if not os.environ.get("OPENAI_API_KEY"):
        raise EnvironmentError("OPENAI_API_KEY is not set.")
    if not os.environ.get("GOOGLE_API_KEY"):
        raise EnvironmentError("GOOGLE_API_KEY is not set.")
    print("API keys loaded from environment variables.")

API keys loaded from Colab Secrets.


In [ ]:
# Set to False to generate fresh embeddings (costs API tokens — recommended for accuracy).
# Set to True to load pre-computed embeddings from file (fast, free, but model-dependent).
load_embedding = False

# Load Dataset


## Download Dataset (JSON)


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv

--2026-04-03 07:23:21--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 173646 (170K) [text/plain]
Saving to: ‘mini-llama-articles.csv’

mini-llama-articles 100%[===================>] 169.58K  --.-KB/s    in 0.006s  

2026-04-03 07:23:22 (27.6 MB/s) - ‘mini-llama-articles.csv’ saved [173646/173646]

--2026-04-03 07:23:22--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP r

## Read File


In [ ]:
# Split the input text into chunks of specified size.
def split_into_chunks(text, chunk_size=1024):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i : i + chunk_size])

    return chunks

In [ ]:
import csv

chunks = []

# Load the file as a CSV
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        chunks.extend(split_into_chunks(row[1]))

In [ ]:
print("number of articles:", idx)
print("number of chunks:", len(chunks))

number of articles: 14
number of chunks: 174


In [ ]:
import pandas as pd

# Convert the JSON list to a Pandas Dataframe
df = pd.DataFrame(chunks, columns=["chunk"])

df.keys()

Index(['chunk'], dtype='object')

# Generate Embedding


In [ ]:
from openai import OpenAI

client = OpenAI()


# Defining a function that converts a text to embedding vector using OpenAI's Ada model.
def get_embedding(text):
    try:
        # Remove newlines
        text = text.replace("\n", " ")
        res = client.embeddings.create(input=[text], model="text-embedding-3-small")

        return res.data[0].embedding

    except:
        return None

In [ ]:
from tqdm import tqdm
import numpy as np

# Generate embedding
if not load_embedding:
    print("Generating embeddings...")
    embeddings = []
    for index, row in tqdm(df.iterrows()):
        # df.at[index, 'embedding'] = get_embedding( row['chunk'] )
        embeddings.append(get_embedding(row["chunk"]))

    embeddings_values = pd.Series(embeddings)
    df.insert(loc=1, column="embedding", value=embeddings_values)

# Or, load the embedding from the file.
else:
    print("Loaded the embedding file.")
    # Load the file as a CSV
    df = pd.read_csv("mini-llama-articles-with_embeddings.csv")
    # Convert embedding column to an array
    df["embedding"] = df["embedding"].apply(lambda x: np.array(eval(x)), 0)

Generating embeddings...


174it [01:01,  2.81it/s]


In [ ]:
# df.to_csv('mini-llama-articles-with_embeddings.csv')

# User Question


In [ ]:
# Define the user question and convert it to an embedding vector.
QUESTION = "How many parameters LLaMA2 model has?"
QUESTION_emb = get_embedding(QUESTION)

print(f"Question   : {QUESTION}")
print(f"Embedding dimensions: {len(QUESTION_emb)}")

Question   : How many parameters LLaMA2 model has?
Embedding dimensions: 1536


# Test Cosine Similarity


Calculating the similarity of embedding representations can help us to find pieces of text that are close to each other. In the following sample you see how the Cosine Similarity metric can identify which sentence could be a possible answer for the given user question. Obviously, the unrelated answer will score lower.


In [ ]:
BAD_SOURCE_emb = get_embedding("The sky is blue.")
GOOD_SOURCE_emb = get_embedding("LLaMA2 model has a total of 2B parameters.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# A sample that how a good piece of text can achieve high similarity score compared
# to a completely unrelated text.
print("> Bad Response Score:", cosine_similarity([QUESTION_emb], [BAD_SOURCE_emb]))
print("> Good Response Score:", cosine_similarity([QUESTION_emb], [GOOD_SOURCE_emb]))

> Bad Response Score: [[0.02581712]]
> Good Response Score: [[0.83164434]]


# Calculate Cosine Similarities


In [ ]:
# The similarity between the questions and each part of the essay.
cosine_similarities = cosine_similarity([QUESTION_emb], df["embedding"].tolist())

print(cosine_similarities)

[[0.46767637 0.46912938 0.25976955 0.29371899 0.31963376 0.40169559
  0.41482155 0.45248033 0.45940843 0.12603983 0.1175329  0.01347136
  0.2261778  0.21449917 0.10144909 0.33114617 0.10739263 0.34692712
  0.1630769  0.0873588  0.34821303 0.22838244 0.19206578 0.26475581
  0.24955873 0.34835557 0.24831962 0.32772256 0.414214   0.41328753
  0.46371324 0.38310087 0.4684696  0.35640921 0.35397384 0.30143728
  0.29939416 0.29257585 0.40025228 0.46468172 0.39479787 0.41044531
  0.44691054 0.43176477 0.35907757 0.33973419 0.51355382 0.20922912
  0.40208173 0.32838125 0.42871227 0.48175007 0.4503181  0.34261757
  0.32082807 0.42600917 0.2469357  0.18071484 0.23654531 0.34267094
  0.34387014 0.20480593 0.19774587 0.22447198 0.21111321 0.42306697
  0.26410728 0.30439882 0.33615852 0.38306147 0.23534345 0.24351726
  0.37072882 0.28031827 0.49050555 0.53056388 0.37837888 0.43773364
  0.37750811 0.39260977 0.30073298 0.41828984 0.4673735  0.45428195
  0.3515488  0.21225359 0.42650355 0.31600601 0.

In [ ]:
import numpy as np

number_of_chunks_to_retrieve = 3

# Sort the scores
highest_index = np.argmax(cosine_similarities)

# Pick the N highest scored chunks
indices = np.argsort(cosine_similarities[0])[::-1][:number_of_chunks_to_retrieve]
print(indices)

[114  75  89]


In [ ]:
# Look at the highest scored retrieved pieces of text
for idx, item in enumerate(df.chunk[indices]):
    print(f"> Chunk {idx+1}")
    print(item)
    print("----")

> Chunk 1
by Meta that ventures into both the AI and academic spaces. The model aims to help researchers, scientists, and engineers advance their work in exploring AI applications. It will be released under a non-commercial license to prevent misuse, and access will be granted to academic researchers, individuals, and organizations affiliated with the government, civil society, academia, and industry research facilities on a selective case-by-case basis. The sharing of codes and weights allows other researchers to test new approaches in LLMs. The LLaMA models have a range of 7 billion to 65 billion parameters. LLaMA-65B can be compared to DeepMind's Chinchilla and Google's PaLM. Publicly available unlabeled data was used to train these models, and training smaller foundational models require less computing power and resources. LLaMA 65B and 33B have been trained on 1.4 trillion tokens in 20 different languages, and according to the Facebook Artificial Intelligence Research (FAIR) team,

# Augment the Prompt


In [ ]:
from google import genai
from google.genai import types
from google.genai.types import HttpOptions

# Initialize the Gemini client using the GOOGLE_API_KEY environment variable.
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# Use the Gemini API to answer the questions based on the retrieved pieces of text.
try:
    # System instructions for the AI assistant
    system_instruction = (
        "You are an assistant and expert in answering questions from a chunks of content. "
        "Only answer AI-related question, else say that you cannot answer this question."
    )

    # Create a user prompt with the retrieved context injected
    prompt = (
        "Read the following information that may contain the context you need to answer the question. "
        "The context is enclosed between <START_OF_CONTEXT> and <END_OF_CONTEXT> tags.\n\n"
        "<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
        "Please provide an informative and accurate answer based on the context above. "
        "Be concise.\nQuestion: {}\nAnswer:"
    )

    # Add the retrieved pieces of text to the prompt
    formatted_prompt = prompt.format("".join(df.chunk[indices]), QUESTION)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=formatted_prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0),
            system_instruction=system_instruction,
            temperature=0.0,
        )
    )

    response = response.text

except Exception as e:
    print(f"An error occurred: {e}")

print(response)

LLaMA 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters.


## How Augmenting the Prompt can address knowledge cutoff limitations and hallucinations

In [ ]:
# Consider this as a retrieved chunk
# https://ai.meta.com/blog/llama-4-multimodal-intelligence (Summarized Content)

Example_chunk ="""
Meta has unveiled the **Llama 4 herd**, a new generation of open-weight, natively multimodal AI models designed to push the boundaries of performance, efficiency, and accessibility. The release includes **Llama 4 Scout** and **Llama 4 Maverick**, alongside a preview of the massive **Llama 4 Behemoth** teacher model. These models introduce advanced **mixture-of-experts (MoE) architectures**, native text–image integration, and record-breaking context lengths, establishing Llama 4 as a major leap in multimodal AI innovation.
**Llama 4 Scout** is a **17B active parameter model with 16 experts** (109B total parameters) that offers **10 million token context length**, far exceeding Llama 3’s 128K limit. Scout’s architecture leverages **interleaved attention layers (iRoPE)** and inference-time temperature scaling to achieve superior length generalization, enabling complex tasks like multi-document summarization, codebase reasoning, and long-context retrieval. Despite its smaller size, Scout surpasses prior Llama models and competitors such as Gemma 3 and Gemini 2.0 Flash-Lite in performance, all while being deployable on a **single NVIDIA H100 GPU**. Its multimodal training allows strong image grounding, multi-image reasoning, and visual question answering.
**Llama 4 Maverick**, also with **17B active parameters**, uses **128 experts** and totals **400B parameters**, alternating dense and MoE layers for inference efficiency. It rivals or exceeds larger models like **GPT-4o, Gemini 2.0 Flash, and DeepSeek v3** on benchmarks for coding, reasoning, multilinguality, and multimodal tasks. Its efficient design makes it deployable on a single **NVIDIA H100 DGX host**, balancing performance with cost-effectiveness. Maverick was refined through a revamped **post-training pipeline**—lightweight supervised fine-tuning, continuous **online reinforcement learning (RL)** with adaptive difficulty filtering, and direct preference optimization—resulting in superior reasoning, conversational fluency, and multimodal understanding. Maverick’s **chat version achieved an ELO score of 1417 on LMArena**, reflecting best-in-class general assistant capabilities.
At the top of the hierarchy, **Llama 4 Behemoth** serves as the **teacher model**, with **288B active parameters, 16 experts, and nearly 2 trillion total parameters**. It outperforms **GPT-4.5, Claude Sonnet 3.7, and Gemini 2.0 Pro** on STEM benchmarks like MATH-500 and GPQA Diamond. Behemoth’s scale demanded innovative training strategies, including pruning 95% of supervised fine-tuning data, advanced reinforcement learning with dynamically filtered prompts, and an optimized asynchronous MoE-parallelized RL infrastructure. These techniques boosted reasoning, coding, and multilingual capabilities while maintaining instruction-following reliability.
All Llama 4 models are **natively multimodal**, trained with large-scale text, image, and video data, and leverage **Meta’s MetaP technique** for reliable hyperparameter scaling. They support **200 languages**, with 10x more multilingual tokens than Llama 3, and employ **FP8 precision** for efficient training across trillions of tokens. Safety remains a priority: Meta integrates **Llama Guard**, **Prompt Guard**, and **CyberSecEval** for content protection, while **Generative Offensive Agent Testing (GOAT)** automates adversarial red-teaming. Llama 4 also significantly reduces **political and social bias**, refusing fewer prompts and responding more neutrally than Llama 3.
In sum, **Llama 4 Scout, Maverick, and Behemoth** represent a major leap in open AI research: compact yet powerful models with unmatched context length, multimodal fluency, reasoning power, and efficiency. By making Scout and Maverick openly available on **llama.com and Hugging Face**, Meta empowers developers, enterprises, and researchers worldwide to build the next generation of AI-driven experiences."""

In [ ]:
QUESTION = "How many parameters does LLaMA 4 model have?"

system_instruction = (
    "You are an assistant and expert in answering questions from a chunks of content. "
    "Only answer AI-related question, else say that you cannot answer this question."
)

# Combining the context with the user's question
prompt = (
    "Read the following information that may contain the context you need to answer the question. "
    "The context is enclosed between <START_OF_CONTEXT> and <END_OF_CONTEXT> tags.\n\n"
    "<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
    "Please provide an informative and accurate answer based on the context above. "
    "Be concise.\nQuestion: {}\nAnswer:"
)

formatted_prompt = prompt.format(Example_chunk, QUESTION)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=formatted_prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction=system_instruction,
        temperature=0.0,
    )
)

response = response.text
print(response)

The Llama 4 models have varying parameters: Llama 4 Scout has 17B active parameters (109B total), Llama 4 Maverick has 17B active parameters (400B total), and Llama 4 Behemoth has 288B active parameters (nearly 2 trillion total).


# Without Augmentation


Test the Gemini API to answer the same question without the addition of retrieved documents. Basically, the LLM will use its knowledge to answer the question.


In [ ]:
# Test without retrieved documents
QUESTION = "How many parameters does LLaMA 4 model have?"

# System instructions
system_instruction = "You are an assistant and expert in answering questions."

# Simple prompt — no retrieved context, LLM answers from memory only
prompt = "Answer the following question concisely.\nQuestion: {}\nAnswer:"
formatted_prompt = prompt.format(QUESTION)

response_no_augment = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=formatted_prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction=system_instruction,
        temperature=0.0,
    )
)

res = response_no_augment.text
print(res)

LLaMA 4 has not been released.


---
## Optional: Visualize Similarity Scores

Instead of just printing the top-3 chunks, we can visualize the similarity scores
across all chunks as a bar chart. This makes it easy to see how much higher the
top-ranked chunks score compared to the rest of the dataset.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — works in scripts and Colab
import matplotlib.pyplot as plt

scores = cosine_similarities[0]
top_n = 10
top_plot_indices = np.argsort(scores)[-top_n:][::-1]
top_plot_scores = scores[top_plot_indices]

# Highlight the actually retrieved chunks (top-3) in green, others in blue
colors = ['#2ecc71' if i in indices else '#3498db' for i in top_plot_indices]

plt.figure(figsize=(10, 4))
plt.bar([f"Chunk {i}" for i in top_plot_indices], top_plot_scores, color=colors)
plt.title("Top 10 Chunks by Cosine Similarity Score")
plt.xlabel("Chunk Index")
plt.ylabel("Cosine Similarity")
plt.xticks(rotation=45, ha='right')

# Add a legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Retrieved (top 3)'),
    Patch(facecolor='#3498db', label='Not retrieved'),
]
plt.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.savefig("similarity_scores.png", dpi=80)
print("Chart saved as similarity_scores.png")
plt.close()

print(f"\nTop {top_n} chunks ranked by similarity:")
print(f"{'Rank':<6} {'Chunk':>8} {'Score':>8}  {'Status'}")
print("-" * 40)
for rank, (idx, score) in enumerate(zip(top_plot_indices, top_plot_scores), 1):
    status = "← retrieved" if idx in indices else ""
    print(f"{rank:<6} {idx:>8} {score:>8.4f}  {status}")

Chart saved as similarity_scores.png

Top 10 chunks ranked by similarity:
Rank      Chunk    Score  Status
----------------------------------------
1           114   0.6195  ← retrieved
2            75   0.5306  ← retrieved
3            89   0.5272  ← retrieved
4            46   0.5136  
5            90   0.5062  
6            91   0.4974  
7            74   0.4905  
8            51   0.4818  
9             1   0.4691  
10           32   0.4685  


---
## Optional: RAG Quality Summary — With vs. Without Augmentation

One of the key benefits of RAG is grounding the model's answer in retrieved facts,
which reduces hallucinations. The cell below stores both responses and prints a
clear side-by-side comparison so you can judge the quality difference directly.

In [ ]:
# Store the final responses from the augmented and non-augmented cells above.
# (These variables are set by the earlier cells in this notebook.)
try:
    augmented_response = response   # Last value of 'res' after the augmented Llama 4 query
    no_aug_response    = response   # This will be overwritten below if cells ran in order
    # Re-run the no-augmentation query to keep responses separate
    _no_aug_prompt = "Answer the following question concisely.\nQuestion: How many parameters does LLaMA 4 model have?\nAnswer:"
    _no_aug_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=_no_aug_prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0),
            system_instruction="You are an expert AI assistant.",
            temperature=0.0,
        )
    )
    no_aug_response = _no_aug_resp.text

    width = 70
    print("RAG QUALITY COMPARISON")
    print("=" * width)
    print(f"\n{'WITH Augmentation (retrieved context injected)':^{width}}")
    print("-" * width)
    print(augmented_response[:600])

    print(f"\n\n{'WITHOUT Augmentation (model memory only)':^{width}}")
    print("-" * width)
    print(no_aug_response[:600])

    print("\n" + "=" * width)
    print("Observation: The augmented response uses specific numbers from the")
    print("retrieved chunks; the non-augmented response may be vague or incorrect.")

except Exception as e:
    print(f"Could not run comparison: {e}")

RAG QUALITY COMPARISON

            WITH Augmentation (retrieved context injected)            
----------------------------------------------------------------------
The Llama 4 models have varying parameters: Llama 4 Scout has 17B active parameters (109B total), Llama 4 Maverick has 17B active parameters (400B total), and Llama 4 Behemoth has 288B active parameters (nearly 2 trillion total).


               WITHOUT Augmentation (model memory only)               
----------------------------------------------------------------------
LLaMA 4 has not been released.

Observation: The augmented response uses specific numbers from the
retrieved chunks; the non-augmented response may be vague or incorrect.
